# Football AI System Demo

This notebook demonstrates the new football AI system that provides comprehensive football video analysis with clean architecture.

In [ ]:
import sys
import os

# Add the football AI system to path
sys.path.append('/workspaces/football_analysis')

from football_ai.pipeline import FootballAnalysisPipeline
import numpy as np

## Configuration

In [ ]:
# Configuration paths
MODEL_PATH = "/workspaces/football_analysis/models/best.pt"
INPUT_VIDEO_PATH = "/workspaces/football_analysis/input_videos/08fd33_4.mp4"
OUTPUT_VIDEO_PATH = "/workspaces/football_analysis/output_videos/football_ai_output.mp4"
OUTPUT_DIR = "/workspaces/football_analysis/football_ai_output"

# Field keypoints for coordinate transformation (optional)
# These should be the four corners of the football field in pixel coordinates
# Format: [top-left, top-right, bottom-right, bottom-left]
FIELD_KEYPOINTS = [
    [110, 1035],  # Top-left corner
    [1640, 915],  # Top-right corner  
    [1770, 570],  # Bottom-right corner
    [240, 390]    # Bottom-left corner
]

print(f"Model path: {MODEL_PATH}")
print(f"Input video: {INPUT_VIDEO_PATH}")
print(f"Output video: {OUTPUT_VIDEO_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

## Initialize the Football AI Pipeline

In [ ]:
# Create the football AI analysis pipeline
pipeline = FootballAnalysisPipeline(
    model_path=MODEL_PATH,
    output_dir=OUTPUT_DIR,
    save_cache=True,
    load_cache=True
)

print("Football AI analysis pipeline initialized successfully!")

## Process the Video

In [ ]:
# Process the football video
print("Starting video analysis...")

try:
    analysis_results = pipeline.process_video(
        video_path=INPUT_VIDEO_PATH,
        output_video_path=OUTPUT_VIDEO_PATH,
        field_keypoints=FIELD_KEYPOINTS
    )
    
    print("\n=== Analysis Complete ===")
    print(f"Total frames processed: {analysis_results.total_frames}")
    print(f"Video FPS: {analysis_results.fps}")
    print(f"Player tracks found: {len(analysis_results.player_tracks)}")
    print(f"Referee tracks found: {len(analysis_results.referee_tracks)}")
    print(f"Ball tracking frames: {len(analysis_results.ball_tracks)}")
    
    # Team possession statistics
    team_1_possession = analysis_results.team_ball_control[0]
    team_2_possession = analysis_results.team_ball_control[1]
    
    print(f"\n=== Team Possession ===")
    print(f"Team 1 possession: {team_1_possession:.1f}%")
    print(f"Team 2 possession: {team_2_possession:.1f}%")
    
except Exception as e:
    print(f"Error during analysis: {e}")
    import traceback
    traceback.print_exc()

## Analysis Results

In [ ]:
# Display detailed analysis results
if 'analysis_results' in locals():
    print("=== Detailed Analysis Results ===")
    
    # Player tracking statistics
    print("\n--- Player Tracking Statistics ---")
    for track_id, player_states in analysis_results.player_tracks.items():
        if len(player_states) > 10:  # Only show players tracked for significant time
            total_distance = sum(state.distance or 0 for state in player_states if state.distance)
            max_speed = max((state.speed or 0 for state in player_states if state.speed), default=0)
            avg_speed = np.mean([state.speed for state in player_states if state.speed]) if any(state.speed for state in player_states) else 0
            
            print(f"Player {track_id}:")
            print(f"  Frames tracked: {len(player_states)}")
            print(f"  Total distance: {total_distance:.1f}m")
            print(f"  Max speed: {max_speed:.1f} km/h")
            print(f"  Avg speed: {avg_speed:.1f} km/h")
            
            # Team assignment
            team_assignments = [state.team for state in player_states if state.team]
            if team_assignments:
                most_common_team = max(set(team_assignments), key=team_assignments.count)
                print(f"  Team: {most_common_team}")
            print()
    
    # Camera movement statistics
    print("--- Camera Movement Statistics ---")
    if analysis_results.camera_movement:
        total_x_movement = sum(abs(movement[0]) for movement in analysis_results.camera_movement)
        total_y_movement = sum(abs(movement[1]) for movement in analysis_results.camera_movement)
        print(f"Total camera X movement: {total_x_movement:.1f} pixels")
        print(f"Total camera Y movement: {total_y_movement:.1f} pixels")
        print(f"Average movement per frame: {(total_x_movement + total_y_movement) / len(analysis_results.camera_movement):.2f} pixels")
    
    print("\n=== Output Files ===")
    print(f"Annotated video saved to: {OUTPUT_VIDEO_PATH}")
    print(f"Analysis cache saved to: {OUTPUT_DIR}/analysis_cache.pkl")
    
else:
    print("No analysis results available. Please run the analysis first.")

## Save Analysis Results

In [ ]:
# Save detailed analysis results
if 'analysis_results' in locals():
    results_path = os.path.join(OUTPUT_DIR, "detailed_analysis_results.pkl")
    pipeline.save_results(results_path)
    print(f"Detailed analysis results saved to: {results_path}")
    
    # Also save a summary as JSON for easy reading
    import json
    
    summary = {
        "total_frames": analysis_results.total_frames,
        "fps": analysis_results.fps,
        "players_tracked": len(analysis_results.player_tracks),
        "referees_tracked": len(analysis_results.referee_tracks),
        "team_1_possession": float(analysis_results.team_ball_control[0]),
        "team_2_possession": float(analysis_results.team_ball_control[1]),
        "camera_movement_frames": len(analysis_results.camera_movement)
    }
    
    summary_path = os.path.join(OUTPUT_DIR, "analysis_summary.json")
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"Analysis summary saved to: {summary_path}")
    print("\\n=== Football AI Analysis Complete ===")

## Compare with Original

This Football AI system provides the same core functionality as the original notebook but with:

1. **Clean Architecture**: Modular components with clear interfaces
2. **Better Error Handling**: Robust error handling and recovery
3. **Extensibility**: Easy to add new analysis features
4. **Clean Code**: Type hints, proper abstractions, and maintainable code
5. **Performance**: Optimized processing pipeline
6. **Caching**: Intelligent caching for faster re-runs

The output video should match the quality and annotations of the original system while being produced by much cleaner, more maintainable code.